<a href="https://colab.research.google.com/github/jxy-ygjm/EC1B1-Coursework/blob/main/EC1B1_Code_ipynb_Group_55.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EC1B1 Coursework

**Winter Term 2025/2026**

<div style="font-family: system-ui; padding: 20px 30px 20px 20px; background-color: #FFFFFF; border-left: 8px solid #0570b0; border-radius: 8px; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1);max-width:600px;color:#212121;">

**Coursework Notebook**
- Group 55
    - Xuanyang Ji
    - Felix Kalu
    - Rishi Ganiger
- Date: 28/02/2026

</div>

**Importing libraries**

To run the notebook, we first need to import all the libraries below

In [1]:
import pandas as pd

## Section 1: Importing the Data

In 24/03/2025, International Financial Statistics (IFS) data were organised around topics with indicators accessible via existing datasets in the IMF Data Portal. IMF deleted the "Query" function, so in this Notebook, we download data from different datasets separately.

The raw data are stored on [GitHub](https://github.com/jxy-ygjm/EC1B1-Coursework), allowing easier data import.

We store all the raw data in a dictionary, so we can easily handle procedures like merge

In [4]:
# Raw URL of data
raw_url = (
    "https://raw.githubusercontent.com/jxy-ygjm/EC1B1-Coursework/main/data/raw/"
)

# Specific data name
data_name = [
    "consumer_price", "exchange_rate", "industrial_production",
    "international_reserves"
]

In [3]:
# Create empty dictionary storing data
data = {}

# Read data
for name in data_name:
    data[name] = pd.read_csv(raw_url + name + ".csv")

Now, we try to investigate the dataframes

In [7]:
for value in data.values():
    print(value.columns)

Index(['COUNTRY', 'INDEX_TYPE', 'COICOP_1999', 'TYPE_OF_TRANSFORMATION',
       'FREQUENCY', 'TIME_PERIOD', 'OBS_VALUE', 'SCALE'],
      dtype='object')
Index(['COUNTRY', 'INDICATOR', 'TYPE_OF_TRANSFORMATION', 'FREQUENCY',
       'TIME_PERIOD', 'OBS_VALUE', 'SCALE'],
      dtype='object')
Index(['COUNTRY', 'PRODUCTION_INDEX', 'TYPE_OF_TRANSFORMATION', 'FREQUENCY',
       'TIME_PERIOD', 'OBS_VALUE', 'SCALE'],
      dtype='object')
Index(['COUNTRY', 'INDICATOR', 'UNIT', 'FREQUENCY', 'TIME_PERIOD', 'OBS_VALUE',
       'SCALE'],
      dtype='object')


We can see that only the `TIME_PERIOD` and `OBS_VALUE` are useful to us.

We now try to create a single dataframe for Spain and US

In [12]:
# Set a list for clean Spain data
spain_cleaned = []

for name, df in data.items():
    temp = df[df["COUNTRY"] == "Spain"][["TIME_PERIOD", "OBS_VALUE"]].copy()
    temp = temp.rename(columns={"OBS_VALUE": name})
    temp = temp.set_index("TIME_PERIOD")
    spain_cleaned.append(temp)

df_spain = pd.concat(spain_cleaned, axis=1)

In [14]:
# Investigate the dataframe
df_spain.head()

,consumer_price,exchange_rate,industrial_production,international_reserves
TIME_PERIOD,,,,
1959-M12,2.525752,60.0,NaN,200.14101
1960-M01,2.505928,60.0,NaN,233.26019
1960-M02,2.503895,60.0,NaN,253.21561
1960-M03,2.500846,60.0,NaN,299.28248
1960-M04,2.502879,60.0,NaN,326.28248


In [15]:
# Set a list for clean US data
us_cleaned = []

for name, df in data.items():
    temp = (
        df[df["COUNTRY"] == "United States"]
        [["TIME_PERIOD", "OBS_VALUE"]]
        .copy()
    )
    temp = temp.rename(columns={"OBS_VALUE": name})
    temp = temp.set_index("TIME_PERIOD")
    us_cleaned.append(temp)

df_us = pd.concat(us_cleaned, axis=1)

In [16]:
# Investigate the dataframe
df_us.head()

,consumer_price,exchange_rate,industrial_production,international_reserves
TIME_PERIOD,,,,
1959-M12,13.482806,NaN,NaN,21543.4138
1960-M01,13.436946,NaN,NaN,21539.3167
1960-M02,13.482806,NaN,NaN,21445.6179
1960-M03,13.482806,NaN,NaN,21411.2592
1960-M04,13.528666,NaN,NaN,21344.4744


Remember no data of `exchange_rate` and `industrial_production` for US, so drop them

In [17]:
df_us = df_us.drop(columns=["exchange_rate", "industrial_production"])

As `consumer_price`, `exchange_rate` and `industrial_production` are indices or have direct meaning, we do not need to know their scale, but for international_reserves, as its nominal measure, we need to know its scale

In [20]:
data["international_reserves"]["SCALE"].head()

,SCALE
0,Millions
1,Millions
2,Millions
3,Millions
4,Millions


Can see all the data are in Millions scale, we can rename the column to highlight this

In [21]:
df_spain = df_spain.rename(
    columns={"international_reserves": "international_reserves (M)"}
)
df_us = df_us.rename(
    columns={"international_reserves": "international_reserves (M)"}
)

## Section 2: Cleaning the Data